# Activation Extraction

**Run once.** Loads the model, extracts activations, and saves everything the analysis notebooks need to `data/v2/activations/`.

| Output | Path | Used by |
|---|---|---|
| Focal-layer activations (layer 13) | `data/v2/activations/*.pt` | braico, replication |
| All-layer sweep | `data/v2/activations/layer_<N>/*.pt` | braico, replication |
| Unembedding covariance | `data/v2/activations/unembed_cov.pt` | braico (Park whitening) |

In [ ]:
import sys
from pathlib import Path

import torch

sys.path.insert(0, str(Path("..").resolve()))  # src/notebooks/ -> src/

from lib.data_typing import load_accepted
from lib.representations import load_model, extract_activations, extract_activations_multilayer

MODEL          = "google/gemma-2-2b"
LAYER          = 13
ACCEPTED_JSONL = Path("../../data/v2/politeness/accepted.jsonl")
ACT_DIR        = Path("../../data/v2/activations")

## 1. Load Data

In [ ]:
samples = load_accepted(ACCEPTED_JSONL)
print(f"Loaded {len(samples)} samples")

## 2. Load Model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")

model, tokenizer = load_model(MODEL, device)

## 3. Focal-Layer Activations (Layer 13)

In [ ]:
extract_activations(samples, model, tokenizer, LAYER, device, out_dir=ACT_DIR)
print(f"Saved focal-layer activations to {ACT_DIR}")

## 4. All-Layer Sweep

In [ ]:
all_layers = list(range(model.config.num_hidden_layers))
extract_activations_multilayer(samples, model, tokenizer, all_layers, device, out_dir=ACT_DIR)
print(f"Saved {len(all_layers)}-layer sweep to {ACT_DIR}/layer_*/")

## 5. Unembedding Covariance (for Park Causal Whitening)

Computes `Cov(γ)` over the full vocabulary and saves it. Used by the braico notebook to build the Park causal inner product without reloading the model.

In [ ]:
U = model.get_output_embeddings().weight.detach()   # (V, D)
D, Vsz, CHUNK = U.shape[1], U.shape[0], 16384

gram    = torch.zeros(D, D, dtype=torch.float64)
col_sum = torch.zeros(D,    dtype=torch.float64)
for s in range(0, Vsz, CHUNK):
    blk = U[s:s + CHUNK].to("cpu", torch.float32)
    gram    += (blk.T @ blk).double()
    col_sum += blk.sum(0).double()

mean_u     = col_sum / Vsz
cov_unembed = gram / Vsz - torch.outer(mean_u, mean_u)

torch.save(cov_unembed, ACT_DIR / "unembed_cov.pt")
print(f"Saved unembedding covariance ({D}×{D}) to {ACT_DIR}/unembed_cov.pt")